# 専門用語の候補を FLR と C-value でスコア付けして抽出する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/term_extraction.ipynb)

NLP クックブック 03-31 ／ 日本語の形態素解析と文分割

専門的な文書から、その分野の用語らしい語（単名詞と複合名詞）を、頻度だけでなく「どれだけ多様な語と結び付くか」や「より長い語の一部としてだけ出ていないか」も考えてスコア付けし、上位から取り出したい。名詞の連続を候補にして、中川・森の FLR と Frantzi らの C-value を自分で実装する。題材は青空文庫の中谷宇吉郎『雪』（1938 年）。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import requests
from bs4 import BeautifulSoup

# 青空文庫の中谷宇吉郎『雪』の XHTML 版（Shift_JIS）。ルビ（読み）と入力者の注記を除いて本文だけにする
url = "https://www.aozora.gr.jp/cards/001569/files/52468_49669.html"
soup = BeautifulSoup(requests.get(url, timeout=60).content.decode("cp932"), "html.parser")
for tag in soup.select("rp, rt, span.notes"):
    tag.decompose()
text = soup.select_one("div.main_text").get_text()
lines = [l.strip() for l in text.splitlines() if l.strip()]
print(len(text), "文字、", len(lines), "行")
print(next(l for l in lines if len(l) > 50)[:100])

77341 文字、 445 行
この本は雪の結晶について私が北海道で行った研究の経過及びその結果をなるべく分りやすく書いたものである。勿論専門の学者の人に読んでもらうつもりは毛頭ないので、ただ自然の色々な現象について正当な理解を持ち


In [3]:
from collections import Counter
from fugashi import Tagger

tagger = Tagger()
STOP = {"こと", "もの", "ところ"}   # unidic-lite では普通名詞-一般になる形式名詞

def is_part(w):
    """複合名詞を作る語か（名詞・接頭辞・接尾辞。数詞、「的」、副詞的・助数詞的な名詞は除く）"""
    f = w.feature
    return (f.pos1 in ("名詞", "接頭辞", "接尾辞") and f.pos2 not in ("数詞", "形状詞的")
            and f.pos3 not in ("副詞可能", "助数詞可能") and w.surface not in STOP)

def noun_seqs(line):
    """名詞の連続を (語, 品詞) のタプルで返す。先頭の接尾辞と末尾の接頭辞は外す"""
    seqs, buf = [], []
    for w in list(tagger(line)) + [None]:
        if w is not None and is_part(w):
            buf.append((w.surface, w.feature.pos1))
            continue
        while buf and buf[0][1] == "接尾辞":
            buf.pop(0)
        while buf and buf[-1][1] == "接頭辞":
            buf.pop()
        if buf:
            seqs.append(tuple(buf))
        buf = []
    return seqs

raw = [s for l in lines for s in noun_seqs(l)]
POS = {w: p for s in raw for w, p in s}
seqs = [tuple(w for w, _ in s) for s in raw]      # 出現ごとの候補（語のタプル）
freq = Counter(seqs)                               # 候補がそのままの形（より長い名詞連続の一部でなく）で出た回数
print("候補の出現:", len(seqs), " 異なり:", len(freq), " うち複合名詞:", sum(len(s) > 1 for s in freq))

候補の出現: 9152  異なり: 2512  うち複合名詞: 794


In [4]:
import math
from collections import defaultdict

# FLR（中川・森）: 単名詞 N の左右に連接する単名詞の異なり数 LN(N), RN(N) から、候補 CN = N1…NL の
#   LR(CN) = (Π (LN(Ni)+1)(RN(Ni)+1)) ^ (1/2L)、FLR(CN) = f(CN) × LR(CN)（f は単独での出現回数）
left, right = defaultdict(set), defaultdict(set)
for s in freq:
    for a, b in zip(s, s[1:]):
        right[a].add(b)
        left[b].add(a)

def lr(s):
    return math.prod((len(left[n]) + 1) * (len(right[n]) + 1) for n in s) ** (1 / (2 * len(s)))

flr = {s: freq[s] * lr(s) for s in freq}
for n in ["結晶", "雪", "研究"]:
    examples = [("".join(s), f) for s, f in freq.most_common() if n in s and len(s) > 1][:4]
    print(f"{n}: 左 {len(left[n])} 種、右 {len(right[n])} 種、LR {lr((n,)):.1f}、単独 {freq[(n,)]} 回、例 {examples}")

# C-value（Frantzi ら）: 2 語以上の候補 a について
#   C(a) = log2|a| × f(a)                          （a がより長い候補に含まれないとき）
#   C(a) = log2|a| × (f(a) − t(a) / c(a))          （t: より長い候補の中での出現回数、c: その長い候補の種類数）
# f(a) はより長い候補の一部としての出現も数える。長い候補から順に、単独の出現回数を部分列に配る
def spans(s):
    for i in range(len(s)):
        for j in range(i + 2, len(s) + 1):
            if (i, j) != (0, len(s)) and POS[s[i]] != "接尾辞" and POS[s[j - 1]] != "接頭辞":
                yield s[i:j]

sub = Counter()
for s in seqs:
    if len(s) > 1:
        sub[s] += 1
        sub.update(spans(s))
t, c, cvalue = Counter(), Counter(), {}
for a in sorted(sub, key=len, reverse=True):
    cvalue[a] = math.log2(len(a)) * (sub[a] - (t[a] / c[a] if c[a] else 0))
    for b in spans(a):
        t[b] += sub[a] - t[a]
        c[b] += 1
print("C-value を付けた候補:", len(cvalue))

結晶: 左 16 種、右 19 種、LR 18.4、単独 452 回、例 [('結晶形', 10), ('側面結晶', 10), ('結晶質', 8), ('平板結晶', 7)]
雪: 左 15 種、右 10 種、LR 13.3、単独 418 回、例 [('雪華', 23), ('人工雪', 20), ('雪華図説', 11), ('牡丹雪', 9)]
研究: 左 3 種、右 9 種、LR 6.3、単独 114 回、例 [('雪華研究', 4), ('研究史', 2), ('研究費', 1), ('研究題目', 1)]
C-value を付けた候補: 1185


In [5]:
import pandas as pd

def top(score, k=20, multi=False):
    items = [s for s in sorted(score, key=score.get, reverse=True) if not multi or len(s) > 1]
    return [f"{''.join(s)} ({score[s]:.0f})" for s in items[:k]]

pd.DataFrame({"頻度（複合名詞）": top(freq, multi=True), "FLR": top(flr), "FLR（複合名詞）": top(flr, multi=True),
              "C-value": top(cvalue)}, index=range(1, 21))

,頻度（複合名詞）,FLR,FLR（複合名詞）,C-value
1,水蒸気 (98),結晶 (8334),水蒸気 (485),水蒸気 (102)
2,顕微鏡 (36),雪 (5545),樹枝状 (197),角板 (45)
3,角板 (34),研究 (721),角板 (173),顕微鏡 (44)
4,落下速度 (24),水蒸気 (485),雪華 (141),雪華 (39)
5,雪華 (23),図 (459),人工雪 (103),樹枝状 (31)
6,十勝岳 (23),形 (292),硝子板 (92),落下速度 (24)
7,過飽和 (22),写真 (275),顕微鏡 (80),十勝岳 (23)
8,樹枝状 (21),観測 (201),結晶形 (78),顕微鏡写真 (22)
9,人工雪 (20),角柱 (201),気象状態 (75),過飽和 (22)
10,硝子板 (18),樹枝状 (197),六花状 (70),人工雪 (20)


- 候補は、この章の「連続する名詞をつなげて複合名詞を抽出する」と同じく名詞の連続（UniDic の短単位をつないだもの）。そこに「こと・もの」などの形式名詞、副詞的に使う名詞（時・上・ため）、助数詞的な名詞（度・年）、数詞、「的」を加えないという条件を足した。「雪の結晶」のように助詞をはさむ用語は取れない。
- FLR は、多くの異なる語と複合語を作る単名詞を重要な構成要素とみなし、その語を含む候補を上げる。「結晶」は左に 16 種、右に 19 種の語が付き（「側面結晶」「平板結晶」「結晶形」「結晶質」など）LR が 18.4、「研究」は左 3 種・右 9 種で 6.3 だった。『雪』では単名詞の「結晶」「雪」が飛び抜けて高く、複合名詞だけで見ると「水蒸気」「樹枝状」「角板」「雪華」「人工雪」が上位に来た。頻度に掛け算するので、頻度の高い一般語（「研究」「図」「写真」）も上位に残る。
- C-value は、長い候補ほど重み (log2|a|) を付け、より長い候補の一部として出た回数も数えたうえで、それを含む長い候補の種類数で割った分を割り引く。決まった長い候補の中にしか出ない部分列は低くなり、いろいろな長い候補に入る語は上がる（「六角板」「小角板」「角板状」に入る「角板」は、単独の頻度 34 より高い 45 になった）。log2 1 = 0 なので単名詞にはスコアが付かず、2 語以上の候補の順位付けに向く。『雪』では長い候補が少ないので、上位は複合名詞の頻度順に近く、3 語の「顕微鏡写真」「雪華図説」は頻度より上がった。
- どちらも正解のない教師なしの順位付けなので、上位を人が見て用語集や辞書の候補にする使い方が基本。分野の文書の頻度を一般の文書（新聞や Wikipedia など）の頻度と比べる方法（TF-IDF や対数尤度比）を組み合わせると、「研究」「写真」のような一般語を下げられる。取り出した用語を MeCab のユーザー辞書に入れるには、この章の「MeCab (fugashi + UniDic) にユーザー辞書を追加する」を参照。
- FLR と C-value をまとめて計算する Python 版の termextract もあるが、PyPI には登録されていない（検証時に `pip install termextract` では入らなかった）。ここでは定義どおりに数十行で実装した。FLR の LR は、ここでは連接の異なり数で数えた（延べ数で数える流儀もある）。
- 『雪』（中谷宇吉郎、1938 年）は著作権の保護期間が満了しており、青空文庫の本文は自由に使える（入力・校正は青空文庫のボランティア）。